In [1]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain.schema import Document
from dotenv import load_dotenv
from sklearn.metrics import classification_report
from transformers import pipeline
import os
import pandas as pd
import numpy as np
import warnings
import re

from tqdm import tqdm

# from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

warnings.filterwarnings("ignore")
load_dotenv()

True

In [2]:
books = pd.read_csv("cleaned_books_data.csv")
books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


In [3]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: object

In [4]:
books["tagged_description"].to_csv("tagged_descriptions.txt", index=False, header=False, sep="\n")

In [5]:
raw_documents = TextLoader("tagged_descriptions.txt").load()
text_splitter = CharacterTextSplitter(
    separator="\n",  # only split where a newline exists
    chunk_size=1,  # large enough so that no text is forcibly split mid-sentence
    chunk_overlap=0,  # no overlap between chunks
)

documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 1
Created a chunk of size 1214, which is longer than the specified 1
Created a chunk of size 373, which is longer than the specified 1
Created a chunk of size 309, which is longer than the specified 1
Created a chunk of size 483, which is longer than the specified 1
Created a chunk of size 482, which is longer than the specified 1
Created a chunk of size 960, which is longer than the specified 1
Created a chunk of size 188, which is longer than the specified 1
Created a chunk of size 843, which is longer than the specified 1
Created a chunk of size 296, which is longer than the specified 1
Created a chunk of size 197, which is longer than the specified 1
Created a chunk of size 881, which is longer than the specified 1
Created a chunk of size 1088, which is longer than the specified 1
Created a chunk of size 1189, which is longer than the specified 1
Created a chunk of size 304, which is longer than the specified 1
Create

In [6]:
print(documents[0])

page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best

In [7]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

'''db_books = Chroma.from_documents(
    documents,
    embedding=embeddings,
    persist_directory="chroma_books_Database",
)

print("vector db created")'''

'db_books = Chroma.from_documents(\n    documents,\n    embedding=embeddings,\n    persist_directory="chroma_books_Database",\n)\n\nprint("vector db created")'

In [8]:
db_books = Chroma(
    persist_directory="chroma_books_Database",
    embedding_function=embeddings,
)

In [9]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k=10)
docs

[Document(id='14698305-27df-4a3d-8133-537803011946', metadata={'source': 'tagged_descriptions.txt'}, page_content='9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='1f188d8a-d046-444f-a7d0-f8fd433a7dd1', metadata={'source': 'tagged_descriptions.txt'}, page_content='9780744578263 Washed up on the beach during a storm, the sea-thing child clings fearfully to the shore until he discovers his true destiny. Suggested level: primary.'),
 Document(id='2a7816c6-352c-419b-a898-0890aceed3f8', metadata={'source': 'tagged_descriptions.txt'}, page_content="9781406957242 This is a reproduction of the original artefact. Generally these books are created from careful scans of the original. This allows us to preserve the book accurately and present it in the way

In [10]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [2]:
def retrieve_semantic_recommendations(
    query: str,
    top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)

    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)].head(top_k)

NameError: name 'pd' is not defined

In [12]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
82,9780027897500,0027897508,The Journey with Grandmother,Edith Unnerstad,Grandmothers,NaN,A Swedish boy long ago accompanies his ... [re...,1960.0,3.75,197.0,3.0,The Journey with Grandmother,9780027897500 A Swedish boy long ago accompani...
228,9780060782139,0060782137,Time For Kids: Butterflies!,Editors of TIME For Kids,Juvenile Nonfiction,http://books.google.com/books/content?id=OdZxn...,"Butterflies There are 20,000 different kinds o...",2006.0,4.00,32.0,20.0,Time For Kids: Butterflies!,"9780060782139 Butterflies There are 20,000 dif..."
270,9780060885373,0060885378,Little House in the Big Woods,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=7JctN...,A year in the life of two young girls growing ...,2007.0,4.18,198.0,193862.0,Little House in the Big Woods,9780060885373 A year in the life of two young ...
692,9780140448009,0140448004,Three Tales,Gustave Flaubert;Roger Whitehouse;Geoffrey Wall,Fiction,http://books.google.com/books/content?id=XFzga...,Features short fiction by the French naturalis...,2005.0,3.71,110.0,3050.0,Three Tales,9780140448009 Features short fiction by the Fr...
713,9780140714555,0140714553,A Midsummer Night's Dream,William Shakespeare,Drama,http://books.google.com/books/content?id=P0Ceh...,"Two pairs of star-crossed lovers, a feuding pa...",2000.0,3.94,144.0,984.0,A Midsummer Night's Dream,9780140714555 Two pairs of star-crossed lovers...
812,9780142302279,0142302279,Dirty Beasts,Roald Dahl,Juvenile Nonfiction,NaN,Poems tell the stories of a smart pig who outw...,2002.0,4.02,32.0,3953.0,Dirty Beasts,9780142302279 Poems tell the stories of a smar...
855,9780143037392,0143037390,The Read-aloud Handbook,Jim Trelease,Language Arts & Disciplines,http://books.google.com/books/content?id=B2_yU...,Explains the importance of reading aloud to ch...,2006.0,4.40,432.0,4122.0,The Read-aloud Handbook,9780143037392 Explains the importance of readi...
938,9780156949606,0156949601,The Waves,Virginia Woolf,Experimental fiction,http://books.google.com/books/content?id=7HEM1...,"Against the background of the sea, the author ...",1959.0,4.14,297.0,19210.0,The Waves,9780156949606 Against the background of the se...
1078,9780241003008,0241003008,The Very Hungry Caterpillar,Eric Carle,Babytime resource,http://books.google.com/books/content?id=DpGEQ...,Eric Carle's children's classic is the story o...,1994.0,4.29,26.0,340101.0,The Very Hungry Caterpillar,9780241003008 Eric Carle's children's classic ...
1288,9780312890216,0312890214,The Starry Rift,James Tiptree,Fiction,NaN,This novel set in the far-future and filled wi...,1994.0,3.82,250.0,220.0,The Starry Rift,9780312890216 This novel set in the far-future...


## Text Classification:

In [13]:
books["categories"].value_counts()

categories
Fiction                      2111
Juvenile Fiction              390
Biography & Autobiography     311
History                       207
Literary Criticism            124
                             ... 
Conspiracies                    1
Brothers and sisters            1
Rock musicians                  1
Community life                  1
Indic fiction (English)         1
Name: count, Length: 479, dtype: int64

In [14]:
print(books["categories"].value_counts().reset_index().query("count > 25"))

                   categories  count
0                     Fiction   2111
1            Juvenile Fiction    390
2   Biography & Autobiography    311
3                     History    207
4          Literary Criticism    124
5                  Philosophy    117
6                    Religion    117
7     Comics & Graphic Novels    116
8                       Drama     86
9         Juvenile Nonfiction     57
10                    Science     56
11                     Poetry     51
12       Literary Collections     50
13       Business & Economics     49
14             Social Science     48
15            Performing Arts     40
16                    Cooking     35
17                 Psychology     33
18                     Travel     32
19                        Art     32
20        Body, Mind & Spirit     32
21          Political Science     30
22           Health & Fitness     28
23                  Self-Help     28
24                  Computers     28
25     Family & Relationships     27


In [15]:
category_mapping = {
    # Fiction
    "Fiction": "Fiction",
    "Drama": "Fiction",
    "Poetry": "Fiction",
    "Comics & Graphic Novels": "Fiction",
    "Literary Collections": "Fiction",
    # Children's books
    "Juvenile Fiction": "Children's Fiction",
    "Juvenile Nonfiction": "Children's Nonfiction",
    # Nonfiction
    "Biography & Autobiography": "Nonfiction",
    "History": "Nonfiction",
    "Literary Criticism": "Nonfiction",
    "Philosophy": "Nonfiction",
    "Religion": "Nonfiction",
    "Science": "Nonfiction",
    "Business & Economics": "Nonfiction",
    "Social Science": "Nonfiction",
    "Performing Arts": "Nonfiction",
    "Cooking": "Nonfiction",
    "Psychology": "Nonfiction",
    "Travel": "Nonfiction",
    "Art": "Nonfiction",
    "Body, Mind & Spirit": "Nonfiction",
    "Political Science": "Nonfiction",
    "Health & Fitness": "Nonfiction",
    "Self-Help": "Nonfiction",
    "Computers": "Nonfiction",
    "Family & Relationships": "Nonfiction",
}

books['simple_category'] = books['categories'].map(category_mapping)

In [16]:
books[~books["simple_category"].isna()]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_category
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,"Tricked once more by his wily half-brother, Ly...",1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079 Tricked once more by his wily ha...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,Joe and his parents are enjoying a summer holi...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 Joe and his parents are enjoying...,Children's Fiction
31,9780007105045,0007105045,Tree and Leaf,John Ronald Reuel Tolkien,Literary Collections,http://books.google.com/books/content?id=aPb_A...,"""The two works 'On fairy-stories' and 'Leaf by...",2001.0,4.09,176.0,2245.0,Tree and Leaf: The Homecoming of Beorhtnoth : ...,"9780007105045 ""The two works 'On fairy-stories...",Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,Rescued from the lockers in which they were le...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969 Rescued from the lockers in whic...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,This book is the story of a young girl obsesse...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 This book is the story of a youn...,Fiction
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Nonfiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction


In [17]:
books["simple_category"].value_counts(dropna=False)

simple_category
Fiction                  2414
Nonfiction               1374
NaN                       962
Children's Fiction        390
Children's Nonfiction      57
Name: count, dtype: int64

## Zero Shot Classification

In [18]:
# Load the zero-shot classification pipeline

fiction_labels = [
    "Fiction",
    "Children's Fiction",
    "Children's Nonfiction",
    "Nonfiction",
]

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")


Device set to use cpu


In [19]:
sequence = books.loc[books["simple_category"] == "Fiction", "description"].reset_index(drop=True)[
    0
]

In [20]:
classifier(sequence, fiction_labels)

{'sequence': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst

| Label                 | Score     | Meaning                                            |
| --------------------- | --------- | -------------------------------------------------- |
| Fiction               | **0.698** | Very likely fiction.                               |
| Nonfiction            | 0.129     | Some thematic overlap (because of realistic tone). |
| Children’s Fiction    | 0.112     | Low probability.                                   |
| Children’s Nonfiction | 0.060     | Very low probability.                              |


In [21]:
max_index = np.argmax(classifier(sequence, fiction_labels)["scores"])
max_index = classifier(sequence, fiction_labels)["labels"][max_index]
max_index

'Fiction'

In [22]:
def generate_predictions(sequence, fiction_labels):
  predictions = classifier(sequence, fiction_labels)
  max_index = np.argmax(predictions["scores"])
  max_label = predictions["labels"][max_index]
  return max_label

# Getting the labels

In [ ]:
actual_cats = []
predicted_cats = []

'''fiction_books = books.loc[
    books["simple_category"] == "Fiction", "description"
].reset_index(drop=True)

for i in tqdm(range(min(300, len(fiction_books)))):
    sequence = fiction_books[i]
    predicted_cats.append(generate_predictions(sequence, fiction_labels))
    actual_cats.append("Fiction")'''

100%|██████████| 300/300 [22:06<00:00,  4.42s/it]


In [ ]:
'''nonfiction_books = books.loc[
    books["simple_category"] == "Nonfiction", "description"
].reset_index(drop=True)

for i in tqdm(range(min(300, len(nonfiction_books)))):
    sequence = nonfiction_books[i]
    predicted_cats.append(generate_predictions(sequence, fiction_labels))
    actual_cats.append("Nonfiction")'''

100%|██████████| 300/300 [23:14<00:00,  4.65s/it]


In [ ]:
'''children_fiction = books.loc[
    books["simple_category"] == "Children's Fiction", "description"
].reset_index(drop=True)

for desc in tqdm(children_fiction.head(300), desc="Children's Fiction"):
    predicted_cats.append(generate_predictions(desc, fiction_labels))
    actual_cats.append("Children's Fiction")'''

Children's Fiction: 100%|██████████| 300/300 [15:33<00:00,  3.11s/it]


In [ ]:
'''children_nonfiction = books.loc[
    books["simple_category"] == "Children's Nonfiction", "description"
].reset_index(drop=True)

for desc in tqdm(children_nonfiction.head(300), desc="Children's Nonfiction"):
    predicted_cats.append(generate_predictions(desc, fiction_labels))
    actual_cats.append("Children's Nonfiction")'''

Children's Nonfiction: 100%|██████████| 57/57 [03:24<00:00,  3.58s/it]


In [ ]:
'''pred_df = pd.DataFrame({"Actual": actual_cats, "Predicted": predicted_cats})
pred_df.to_csv("category_predictions.csv", index=False)
print("Saved main predictions to category_predictions.csv")

pred_df = pd.read_csv("category_predictions.csv")

pred_df["correct_predictions"] = pred_df["Actual"] == pred_df["Predicted"]
accuracy = pred_df["correct_predictions"].mean()
print(f"Overall accuracy: {accuracy:.2%}")

print(classification_report(pred_df["Actual"], pred_df["Predicted"]))'''

✅ Saved main predictions to category_predictions.csv


In [23]:
pred_df = pd.read_csv("category_predictions_2class.csv")

"""
def simplify(label):
    if "Fiction" in label:
        return "Fiction"
    else:
        return "Nonfiction"


pred_df["Actual_2cat"] = pred_df["Actual"].apply(simplify)
pred_df["Predicted_2cat"] = pred_df["Predicted"].apply(simplify)
"""

print(classification_report(pred_df["Actual_2cat"], pred_df["Predicted_2cat"]))

              precision    recall  f1-score   support

     Fiction       0.88      0.61      0.72       600
  Nonfiction       0.57      0.86      0.68       357

    accuracy                           0.70       957
   macro avg       0.72      0.73      0.70       957
weighted avg       0.76      0.70      0.71       957



In [24]:
isbns = []

predicted_cats = []

missing_cats = books.loc[books["simple_category"].isna(), ["isbn13", "description"]].reset_index(drop=True)

for i in tqdm (range(0, len(missing_cats))):
    sequence = missing_cats.loc[i, "description"]
    predicted_cats.append(generate_predictions(sequence, fiction_labels))
    isbns.append(missing_cats.loc[i, "isbn13"])

100%|██████████| 962/962 [1:06:55<00:00,  4.17s/it]


In [25]:
missing_cats_df = pd.DataFrame({"isbn13": isbns, "Predicted_Category": predicted_cats})
missing_cats_df

books = pd.merge(books, missing_cats_df, on="isbn13", how="left")
books["simple_category"] = books["simple_category"].fillna(books["Predicted_Category"])

# ❌ Don't assign when using inplace=True
books.drop(columns=["Predicted_Category"], inplace=True)

books.to_csv("books_with_filled_categories_4way.csv", index=False)
print("✅ Missing categories filled and saved.")

✅ Missing categories filled and saved.


In [26]:
def simplify(label):
    if "Fiction" in str(label):  # catches "Children's Fiction" and "Fiction"
        return "Fiction"
    else:
        return "Nonfiction"

books["simple_category"] = books["simple_category"].apply(simplify)
books.to_csv("books_with_filled_categories_2way.csv", index=False)
print(
    "✅ Simplified to 2 categories and saved as books_with_filled_categories_2way.csv"
)

✅ Simplified to 2 categories and saved as books_with_filled_categories_2way.csv


In [27]:
books["simple_category"].value_counts()

simple_category
Fiction       3217
Nonfiction    1980
Name: count, dtype: int64

In [30]:
books[books["simple_category"] == "Nonfiction"]["categories"].value_counts().head(20)

categories
Biography & Autobiography    311
History                      207
Literary Criticism           124
Religion                     117
Philosophy                   117
Juvenile Nonfiction           57
Science                       56
Business & Economics          49
Social Science                48
Performing Arts               40
Cooking                       35
Psychology                    33
Art                           32
Body, Mind & Spirit           32
Travel                        32
Political Science             30
Self-Help                     28
Computers                     28
Health & Fitness              28
Family & Relationships        27
Name: count, dtype: int64

In [31]:
books[books["simple_category"] == "Fiction"]["categories"].value_counts().head(20)

categories
Fiction                          2111
Juvenile Fiction                  390
Comics & Graphic Novels           116
Drama                              86
Poetry                             51
Literary Collections               50
Children's stories                 11
Humor                               9
Adventure stories                   9
Language Arts & Disciplines         8
English fiction                     8
City and town life                  7
Fantasy fiction                     7
Young Adult Fiction                 7
Detective and mystery stories       7
Science fiction                     6
Life on other planets               6
American fiction                    6
FICTION                             6
London (England)                    5
Name: count, dtype: int64